# King County: preprocessing working copy

T05 starting point, not a finished refactor. These cells come from the original notebook; T06/T07 will replace their inline transformations with imports from our Python package. Run from the repository root.

EDA, model training/evaluation, and model saving remain in the original notebook for now. T08 will connect the working notebook to the complete pipeline. This copy does not train or overwrite any model.

See [the baseline reference](docs/notebook-baseline.md).

In [ ]:
import numpy as np
import pandas as pd

## Load the original dataset

In [ ]:
# Load the dataset with pandas
kc_data = pd.read_csv("data/King_County_House_prices_dataset.csv")

## Training-only row exclusion

In [ ]:
# Drop the implausible row.
kc_data.drop(15856, axis=0, inplace=True)

## Data cleaning

In [ ]:
# Recalculate `sqft_basement` as `sqft_living - sqft_above`.
kc_data["sqft_basement"] = kc_data["sqft_living"] - kc_data["sqft_above"]

In [ ]:
# Replace missing values in `view` with the most frequent value (0).
kc_data["view"] = kc_data["view"].fillna(0)

In [ ]:
# Replace missing values in `waterfront` with the most frequent value (0).
kc_data["waterfront"] = kc_data["waterfront"].fillna(0)

In [ ]:
# Create an empty list to store the derived values.
last_known_change = []

# Inspect the `yr_renovated` value for each row.
for idx, yr_re in kc_data["yr_renovated"].items():
    # If `yr_renovated` is 0 or missing, use the original build year instead.
    if str(yr_re) == "nan" or yr_re == 0.0:
        last_known_change.append(kc_data["yr_built"][idx])
    # Otherwise, keep the renovation year in the new list.
    else:
        last_known_change.append(int(yr_re))

In [ ]:
# Create a new column from the values collected above.
kc_data["last_known_change"] = last_known_change

In [ ]:
# Drop the original `yr_renovated` and `yr_built` columns.
kc_data.drop("yr_renovated", axis=1, inplace=True)
kc_data.drop("yr_built", axis=1, inplace=True)

## Distance to the notebook's fixed centre

In [ ]:
# Absolute difference in latitude between the center and the property.
kc_data["delta_lat"] = np.absolute(47.62774 - kc_data["lat"])
# Absolute difference in longitude between the center and the property.
kc_data["delta_long"] = np.absolute(-122.24194 - kc_data["long"])
# Distance between the center and the property.
kc_data["center_distance"] = (
    (
        (kc_data["delta_long"] * np.cos(np.radians(47.6219))) ** 2
        + kc_data["delta_lat"] ** 2
    )
    ** (1 / 2)
    * 2
    * np.pi
    * 6378
    / 360
)

## Distance to waterfront reference houses

In [ ]:
# This helper function calculates the distance between one house and a reference location.
def dist(long, lat, ref_long, ref_lat):
    """dist computes the distance in km to a reference location.
    Input: long and lat of the location of interest and ref_long and ref_lat
    as the long and lat of the reference location"""
    delta_long = long - ref_long
    delta_lat = lat - ref_lat
    delta_long_corr = delta_long * np.cos(np.radians(ref_lat))
    return (
        ((delta_long_corr) ** 2 + (delta_lat) ** 2) ** (1 / 2) * 2 * np.pi * 6378 / 360
    )

In [ ]:
# Add all waterfront houses to the reference list.
water_list = kc_data.query("waterfront == 1")
water_list.head()

In [ ]:
water_distance = []
# For each row, calculate the distance to the nearest waterfront house.
for idx in kc_data.index:
    ref_list = []
    for x, y in zip(list(water_list["long"]), list(water_list["lat"])):
        ref_list.append(dist(kc_data["long"][idx], kc_data["lat"][idx], x, y).min())
    water_distance.append(min(ref_list))

In [ ]:
# Create a new column from the previously computed list.
kc_data["water_distance"] = water_distance

## Select the model inputs

In [ ]:
# Remove columns that would cause data leakage or add no predictive value.
drop_lst = [
    "price",
    "sqft_price",
    "date",
    "delta_lat",
    "delta_long",
]

In [ ]:
# Keep all remaining variables as candidate features.
all_features = [x for x in kc_data.columns if x not in drop_lst]

In [ ]:
# `X` contains the descriptive variables defined above.
X = kc_data[all_features]

In [ ]:
# The original drops "id" before fitting the final model (cell 102).
X_prepared = X.drop(columns=["id"])
print(f"Prepared rows: {len(X_prepared)}; features: {X_prepared.shape[1]}")
X_prepared.head()